# Data quality

Exploration to choose the cleaning rules before route extraction (`02_route_extraction.ipynb`).
Each section measures one problem on the real data and ends with a **Decision** cell to fill in:

1. MMSI: real vessels vs base stations, aids to navigation, invalid ids
2. Impossible jumps: implied speed between consecutive reports
3. Gaps: long silences that should split a track into separate voyages
4. "Stationary" threshold: how waypoint candidates and waypoints change with it
5. Route coverage: how much traffic ends up in a route, with voyages split at gaps and loops kept

Once a rule is decided, it moves into `ais_anomaly` with a test and notebook 02 uses it.
Figures are saved to `outputs/`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ais_anomaly import routes
from ais_anomaly.geo import haversine
from ais_anomaly.io import read_noaa, save_figure

DATA = Path("../data")
KN = 0.514444  # m/s per knot

## Load

The merged quarter if the download has finished, otherwise the days already downloaded.

In [ ]:
merged = DATA / "hawaii_2020Q1.parquet"
if merged.exists():
    ais = read_noaa(merged)
else:
    days = sorted((DATA / "raw" / "hawaii_2020Q1").glob("AIS_*.parquet"))
    print(f"download not finished: using {len(days)} days ({days[0].stem} .. {days[-1].stem})")
    ais = read_noaa(days[0]) if len(days) == 1 else pd.concat([read_noaa(p) for p in days], ignore_index=True)

ais = ais.sort_values(["MMSI", "BaseDateTime"], ignore_index=True)
print(f"{len(ais):,} reports, {ais['MMSI'].nunique():,} MMSI, {ais['BaseDateTime'].min()} -> {ais['BaseDateTime'].max()}")

In [ ]:
per_day = ais.set_index("BaseDateTime").resample("D").agg({"MMSI": ["size", "nunique"]})
per_day.columns = ["reports", "vessels"]
ax = per_day.plot(subplots=True, figsize=(10, 4), marker="o", title=["Reports per day", "Distinct MMSI per day"])
plt.tight_layout()
save_figure(plt.gcf(), "03_reports_per_day")

## 1. MMSI

A ship's MMSI has 9 digits and starts with its country code (MID, first digit 2-7).
Other ranges are not ships: `00…` coast/base stations, `111…` SAR aircraft, `970-974…` survival beacons,
`98…` craft attached to a ship, `99…` aids to navigation (buoys, lighthouses). Ids like `0` or `123456789` are invalid.
Non-ships are a problem for waypoints: a buoy is a perfectly "stationary" point.

In [ ]:
def mmsi_category(mmsi):
    s = pd.Series(mmsi).astype("int64").astype(str)
    cat = pd.Series("invalid", index=s.index)
    nine = s.str.len() == 9
    cat[nine & s.str[0].isin(list("234567"))] = "ship"
    cat[(s.str.len() == 7) | (nine & s.str.startswith("00"))] = "base station"  # 00MIDxxxx loses its leading zeros
    cat[nine & s.str.startswith("111")] = "SAR aircraft"
    cat[nine & s.str[:3].isin(["970", "972", "974"])] = "beacon"
    cat[nine & s.str.startswith("98")] = "craft of a ship"
    cat[nine & s.str.startswith("99")] = "aid to navigation"
    cat[s.isin(["0", "123456789", "111111111", "999999999"])] = "invalid"
    return cat.to_numpy()


ais["mmsi_cat"] = mmsi_category(ais["MMSI"])
summary = ais.groupby("mmsi_cat").agg(reports=("MMSI", "size"), mmsi=("MMSI", "nunique"))
summary["% reports"] = (100 * summary["reports"] / len(ais)).round(2)
summary.sort_values("reports", ascending=False)

In [ ]:
# what the non-ship ids look like, and the vessel types of real ships
display(ais[ais["mmsi_cat"] != "ship"].groupby(["mmsi_cat", "MMSI"]).agg(
    reports=("LAT", "size"), name=("VesselName", "first"), lat_std=("LAT", "std"), lon_std=("LON", "std")).head(20))
ais.loc[ais["mmsi_cat"] == "ship", "VesselType"].value_counts(dropna=False).head(15)

**Decision (MMSI):** _e.g. keep only `ship`; keep/drop specific VesselType codes …_

## 2. Impossible jumps

Speed implied by two consecutive reports of the same MMSI (haversine distance / time).
Ships rarely exceed ~40 kn; hundreds of knots mean a GPS glitch or two transmitters sharing an MMSI.

In [ ]:
same = ais["MMSI"].eq(ais["MMSI"].shift())
dt = ais["BaseDateTime"].diff().dt.total_seconds()
dist = haversine(ais["LAT"].shift(), ais["LON"].shift(), ais["LAT"], ais["LON"])
ais["dt_s"] = dt.where(same)
ais["implied_kn"] = (dist / dt).where(same & (dt > 0)) / KN

v = ais["implied_kn"].dropna()
print(f"pairs: {len(v):,}")
for thr in (30, 50, 100, 500):
    print(f"  implied speed > {thr:>3} kn: {(v > thr).sum():>8,}  ({100 * (v > thr).mean():.3f} %)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.log10(v[v > 0]), bins=200)
ax.set(xlabel="log10 implied speed [kn]", ylabel="pairs", title="Implied speed between consecutive reports")
for thr in (30, 50, 100):
    ax.axvline(np.log10(thr), color="k", ls="--", lw=1)
save_figure(fig, "03_implied_speed")

In [ ]:
# implied speed vs reported SOG: glitches have a large implied speed and a normal SOG
pairs = ais[ais["implied_kn"].notna() & (ais["SOG"] < 102.3)]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pairs["SOG"], pairs["implied_kn"].clip(upper=200), s=1, alpha=0.2)
ax.plot([0, 40], [0, 40], "r--", lw=1)
ax.set(xlabel="SOG [kn]", ylabel="implied speed [kn] (clipped at 200)", title="Reported vs implied speed")
save_figure(fig, "03_sog_vs_implied")

worst = ais.nlargest(10, "implied_kn")[["MMSI", "VesselName", "BaseDateTime", "LAT", "LON", "SOG", "dt_s", "implied_kn"]]
worst

**Decision (jumps):** _e.g. drop reports whose implied speed from the previous one is > 50 kn …_

## 3. Gaps

Time between consecutive reports of the same MMSI. Data are filtered to one report per minute, so 60 s is normal;
long silences mean the vessel left the area, switched AIS off, or was out of receiver range.
Today a track is never split, so a gap merges separate voyages and entry/exit points are only the first and last report.

In [ ]:
g = ais["dt_s"].dropna() / 60  # minutes
for thr in (5, 30, 60, 180, 360, 1440):
    print(f"  gap > {thr:>4} min: {(g > thr).sum():>7,} pairs  ({100 * (g > thr).mean():.3f} %)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.log10(g[g > 0]), bins=200)
ax.set(xlabel="log10 gap [min]", ylabel="pairs", title="Time between consecutive reports")
for thr in (60, 180, 1440):
    ax.axvline(np.log10(thr), color="k", ls="--", lw=1)
save_figure(fig, "03_gaps")

In [ ]:
# how many voyages per vessel for each split threshold
ships = ais[ais["mmsi_cat"] == "ship"]
rows = []
for thr_min in (30, 60, 180, 360, 1440):
    new_voyage = ~ships["MMSI"].eq(ships["MMSI"].shift()) | (ships["dt_s"] > thr_min * 60)
    voyage_id = new_voyage.cumsum()
    sizes = voyage_id.value_counts()
    rows.append({"split gap [min]": thr_min, "voyages": len(sizes), "voyages >= 50 reports": int((sizes >= 50).sum()),
                 "median reports/voyage": int(sizes.median())})
pd.DataFrame(rows)

**Decision (gaps):** _e.g. split a track when two reports are more than 60 min apart; keep voyages with ≥ 50 reports …_

## 4. "Stationary" threshold

A report is stationary only if the speed computed from positions is exactly 0: a deliberate choice that keeps the
number of waypoint candidates, and the DBSCAN cost, low. Moored vessels jittering by a few metres are therefore not
candidates. Here we measure what larger thresholds would change (candidates, and waypoints found with the parameters
of notebook 02) and whether the computation holds.

In [ ]:
vessels = [routes.add_speed_status(v) for v in routes.split_by_vessel(ais[ais["mmsi_cat"] == "ship"], min_points=50)]
speed = pd.concat(v["avg_speed"] for v in vessels) / KN

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(speed[speed < 3], bins=300)
ax.set(xlabel="speed from positions [kn]", ylabel="reports", title="Low speeds: moored and drifting vessels")
save_figure(fig, "03_low_speeds")
print(f"exactly 0: {(speed == 0).mean():.1%} of reports, < 0.5 kn: {(speed < 0.5).mean():.1%}")

In [ ]:
rows, hull_sets = [], {}
for thr_kn in (0, 0.1, 0.25, 0.5, 1.0):
    vs = []
    for v in vessels:
        v = v.copy()
        v["status"] = np.where(v["avg_speed"] <= thr_kn * KN, "stationary", "sailing")
        vs.append(v)
    cand = routes.waypoint_candidates(vs)
    _, hulls = routes.cluster_waypoints(cand, eps=0.2, min_samples=20)
    hull_sets[thr_kn] = hulls
    rows.append({"threshold [kn]": thr_kn, "candidates": len(cand), "waypoints": len(hulls)})
pd.DataFrame(rows)

In [ ]:
# degenerate clusters (aligned or identical points) have a line/point hull: geopandas draws every geometry type
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
for ax, (thr_kn, hulls) in zip(axes.flat, hull_sets.items()):
    gpd.GeoSeries(hulls["hull"]).plot(ax=ax, alpha=0.5, edgecolor="k", markersize=25, cmap="tab20")
    for name, hull in zip(hulls["name"], hulls["hull"]):
        ax.annotate(name, (hull.centroid.x, hull.centroid.y), fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.set(title=f"stationary <= {thr_kn} kn: {len(hulls)} waypoints", xlabel="LON", ylabel="LAT")
axes.flat[-1].axis("off")
plt.tight_layout()
save_figure(fig, "03_waypoints_by_threshold")

**Decision (stationary):** _e.g. stationary when speed from positions ≤ 0.5 kn …_

## 5. Route coverage

How much of the traffic at sea ends up in a route, and what is lost. Three configurations on the same ships:

- **current**: one track per MMSI, routes only between two different waypoints (what notebook 02 does today);
- **voyages**: tracks cut at gaps longer than `GAP` (`split_by_vessel(max_gap=GAP)`), so each voyage has its own entry/exit point and no route spans a silence;
- **voyages + loops**: also keep trips that leave a waypoint and return to it (`extract_routes(loops=True)`), e.g. fishing trips.

Waypoints are recomputed for each configuration, because entry/exit candidates change with the voyages.
A *pass-through* is a stay inside a waypoint without any stationary report: the vessel only crossed the polygon,
but today it still counts as a visit and splits the track. It depends on the stationary rule (exactly 0 m/s).

In [ ]:
GAP = "60min"  # from section 3
FISHING = 30   # AIS VesselType code


def route_audit(vessels, hulls, loops):
    """Coverage numbers for one configuration; adds the 'waypoints' column to each vessel."""
    at_sea = in_route = loop_reports = n_loops = passes = stops = gaps_60 = 0
    found = []
    for v in vessels:
        v["waypoints"] = routes.assign_waypoints(v, hulls)
        wp = v["waypoints"].to_numpy()
        at_sea += int((wp == routes.NO_WAYPOINT).sum())
        run = np.cumsum(np.r_[True, wp[1:] != wp[:-1]])
        runs = v.groupby(run).agg(wp=("waypoints", "first"), n=("waypoints", "size"),
                                  stop=("status", lambda s: (s == "stationary").any()))
        inside = runs[runs["wp"] != routes.NO_WAYPOINT]
        passes += int((~inside["stop"]).sum())
        stops += int(inside["stop"].sum())
        w = runs["wp"].to_numpy()
        for i in range(1, len(runs) - 1):
            if w[i] == routes.NO_WAYPOINT and w[i - 1] == w[i + 1]:
                n_loops += 1
                loop_reports += int(runs["n"].iloc[i])
        for r in routes.extract_routes(v, loops=loops):
            in_route += len(r) - 1  # minus the report in the start waypoint
            gaps_60 += int(r["BaseDateTime"].diff().max() > pd.Timedelta("60min"))
            found.append(r.assign(VesselType=v["VesselType"].iloc[0]))
    route_df = pd.concat(found) if found else pd.DataFrame(columns=["Route", "VesselType"])
    fishing_routes = sum(int(r["VesselType"].iloc[0] == FISHING) for r in found)
    return {
        "waypoints": len(hulls),
        "reports at sea": at_sea,
        "in a route": f"{in_route / max(at_sea, 1):.1%}",
        "loops (same WP)": n_loops,
        "loop reports": f"{loop_reports / max(at_sea, 1):.1%}",
        "routes": len(found),
        "fishing routes": fishing_routes,
        "routes with gap > 60 min": gaps_60,
        "pass-through visits": passes,
        "visits with a stop": stops,
    }, route_df

In [ ]:
ships = ais[ais["mmsi_cat"] == "ship"]
configs = {"current": (None, False), "voyages": (GAP, False), "voyages + loops": (GAP, True)}
results, route_tables = {}, {}
for name, (max_gap, loops) in configs.items():
    vessels = [routes.add_speed_status(v) for v in routes.split_by_vessel(ships, min_points=50, max_gap=max_gap)]
    _, hulls = routes.cluster_waypoints(routes.waypoint_candidates(vessels), eps=0.2, min_samples=20)
    results[name], route_tables[name] = route_audit(vessels, hulls, loops)
pd.DataFrame(results)

In [ ]:
# which vessel types end up in routes, per configuration
types = pd.DataFrame({name: t.groupby("VesselType")["Route"].nunique() for name, t in route_tables.items()}).fillna(0).astype(int)
ax = types.plot.bar(figsize=(10, 4), title="Distinct routes per vessel type (30 = fishing, 31 = towing, 60 = passenger)")
ax.set(xlabel="VesselType", ylabel="distinct routes")
save_figure(ax.figure, "03_routes_by_vessel_type")
types

**Decision (routes):** _e.g. split voyages at 60 min; keep loops as routes 'WP_a-WP_a'; require a stop to count a port visit …_

## Summary

| Rule | Decision | Effect measured above |
|---|---|---|
| MMSI | | |
| Jumps | | |
| Gaps | | |
| Stationary | | |
| Voyages (gap split) | | |
| Loops | | |
| Port visit needs a stop | | |

Next: implement the chosen rules in `ais_anomaly` (with tests) and rerun `02_route_extraction.ipynb`.